<a href="https://colab.research.google.com/github/hyoungjun0118/-/blob/main/2355004_%EA%B9%80%ED%98%95%EC%A4%80_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install fastapi uvicorn httpx beautifulsoup4 gradio pyngrok nest-asyncio pandas

In [4]:
import sqlite3
import httpx
from bs4 import BeautifulSoup
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import gradio as gr
from pyngrok import ngrok
import nest_asyncio
import uvicorn
from collections import Counter
import pandas as pd
import warnings

# 경고 메시지 숨기기
warnings.filterwarnings('ignore')

In [5]:
def init_db():
    conn = sqlite3.connect("quotes.db")
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS quotes
                      (id INTEGER PRIMARY KEY AUTOINCREMENT,
                       text TEXT, author TEXT, tags TEXT)''')
    conn.commit()
    conn.close()

def scrape_quotes():
    conn = sqlite3.connect("quotes.db")
    cursor = conn.cursor()

    # 이미 20개 이상의 데이터가 있다면 중복 크롤링 방지
    cursor.execute("SELECT COUNT(*) FROM quotes")
    if cursor.fetchone()[0] >= 50:
        conn.close()
        return

    url = "https://quotes.toscrape.com"
    quotes_data = []
    page = 1

    # 총 20개의 명언을 수집할 때까지 페이지 이동
    while len(quotes_data) < 50:
        response = httpx.get(f"{url}/page/{page}/")
        soup = BeautifulSoup(response.text, 'html.parser')
        items = soup.select(".quote")

        for item in items:
            if len(quotes_data) >= 50: break
            text = item.find(class_="text").get_text(strip=True)
            author = item.find(class_="author").get_text(strip=True)
            # 카테고리(태그) 수집
            tags = [tag.get_text(strip=True) for tag in item.select(".tag")]
            quotes_data.append((text, author, ", ".join(tags)))
        page += 1

    cursor.executemany("INSERT INTO quotes (text, author, tags) VALUES (?, ?, ?)", quotes_data)
    conn.commit()
    conn.close()

In [6]:
app = FastAPI(title="Quotes API", description="명언 관리 시스템 API (중간고사 과제)")

class QuoteCreate(BaseModel):
    text: str
    author: str
    tags: str

class QuoteUpdate(BaseModel):
    text: str = None
    author: str = None
    tags: str = None

# C(reate)
@app.post("/quotes", tags=["Quotes"])
def create_quote(quote: QuoteCreate):
    conn = sqlite3.connect("quotes.db")
    cursor = conn.cursor()
    cursor.execute("INSERT INTO quotes (text, author, tags) VALUES (?, ?, ?)",
                   (quote.text, quote.author, quote.tags))
    conn.commit()
    new_id = cursor.lastrowid
    conn.close()
    return {"message": "명언 추가 완료", "id": new_id}

# R(ead)
@app.get("/quotes", tags=["Quotes"])
def read_quotes():
    conn = sqlite3.connect("quotes.db")
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    rows = cursor.execute("SELECT * FROM quotes").fetchall()
    conn.close()
    return [dict(row) for row in rows]

# U(pdate)
@app.put("/quotes/{quote_id}", tags=["Quotes"])
def update_quote(quote_id: int, quote: QuoteUpdate):
    conn = sqlite3.connect("quotes.db")
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM quotes WHERE id = ?", (quote_id,))
    if not cursor.fetchone():
        conn.close()
        raise HTTPException(status_code=404, detail="해당 명언을 찾을 수 없습니다.")

    cursor.execute("UPDATE quotes SET text = COALESCE(?, text), author = COALESCE(?, author), tags = COALESCE(?, tags) WHERE id = ?",
                   (quote.text, quote.author, quote.tags, quote_id))
    conn.commit()
    conn.close()
    return {"message": "명언 수정 완료"}

# D(elete)
@app.delete("/quotes/{quote_id}", tags=["Quotes"])
def delete_quote(quote_id: int):
    conn = sqlite3.connect("quotes.db")
    cursor = conn.cursor()
    cursor.execute("DELETE FROM quotes WHERE id = ?", (quote_id,))
    if cursor.rowcount == 0:
        conn.close()
        raise HTTPException(status_code=404, detail="해당 명언을 찾을 수 없습니다.")
    conn.commit()
    conn.close()
    return {"message": "명언 삭제 완료"}

In [7]:
def get_analysis_data(search_tag=""):
    conn = sqlite3.connect("quotes.db")
    # 카테고리(태그) 필터링 기능 적용
    if search_tag:
        query = f"SELECT * FROM quotes WHERE tags LIKE '%{search_tag}%'"
    else:
        query = "SELECT * FROM quotes"

    df = pd.read_sql_query(query, conn)
    conn.close()

    if df.empty:
        return df, pd.DataFrame(), pd.DataFrame()

    # 1. 단어 빈도수(Word Count) 분석 (기본 요구사항)
    import re
    all_text = " ".join(df['text']).lower()
    words = re.findall(r'\b[a-z]{4,}\b', all_text) # 4글자 이상 단어만 추출
    word_counts = Counter(words).most_common(10)
    df_word_counts = pd.DataFrame(word_counts, columns=['Word', 'Count'])

    # 2. 저자별 명언 수 분석 (추가 가산점 기능)
    author_counts = df['author'].value_counts().reset_index()
    author_counts.columns = ['Author', 'Count']

    return df, df_word_counts, author_counts

with gr.Blocks(theme=gr.themes.Default()) as demo:
    gr.Markdown("# 🎓 중간고사 과제: FastAPI 기반 명언 관리 시스템")
    gr.Markdown("크롤링 데이터, 단어 빈도수, 그리고 카테고리 검색 및 저자 분포를 확인할 수 있습니다.")

    with gr.Row():
        search_input = gr.Textbox(label="🔍 카테고리(태그) 검색 (예: love, life)", placeholder="태그를 입력하고 새로고침을 누르세요")
        btn_refresh = gr.Button("데이터 조회 및 시각화 🚀", variant="primary")

    with gr.Row():
        table = gr.Dataframe(label="수집된 명언 데이터베이스 (SQLite)", interactive=False)

    with gr.Row():
        # 기본 요구사항 시각화
        plot_word = gr.BarPlot(x="Word", y="Count", title="상위 단어 빈도수 (Word Count)",
                               tooltip=["Word", "Count"], color="Word", vertical=False)
        # 추가 기능 시각화
        plot_author = gr.BarPlot(x="Author", y="Count", title="저자별 명언 등록 수",
                                 tooltip=["Author", "Count"], color="Author")

    btn_refresh.click(fn=get_analysis_data, inputs=[search_input], outputs=[table, plot_word, plot_author])

# FastAPI에 Gradio Mount
app = gr.mount_gradio_app(app, demo, path="/ui")

new /ui


In [8]:
if __name__ == "__main__":
    # DB 초기화 및 20개 크롤링 실행
    init_db()
    scrape_quotes()

    # 알려주신 토큰 적용 완료!
    NGROK_TOKEN = "3ChTmV0LuwFgxG0QlwxRqGbzvGZ_64cyaucWGFjK6uQBqbgGk"

In [9]:
ngrok.kill()

In [10]:
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000)

print("\n" + "🚀" * 30)
print(f"📖 Swagger API 명세서: {public_url.public_url}/docs")
print(f"🎨 Gradio 사용자 UI: {public_url.public_url}/ui")
print("🚀" * 30 + "\n")
print("위 링크를 클릭하여 과제 결과물을 확인하세요!")


🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
📖 Swagger API 명세서: https://populace-drove-refurnish.ngrok-free.dev/docs
🎨 Gradio 사용자 UI: https://populace-drove-refurnish.ngrok-free.dev/ui
🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀

위 링크를 클릭하여 과제 결과물을 확인하세요!


In [11]:
import asyncio

# 코랩 환경용 비동기 처리 적용
nest_asyncio.apply()

# uvicorn.run 대신 Server 객체를 사용하여 루프 충돌 방지 및 백그라운드 실행
async def run_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
    server = uvicorn.Server(config)
    await server.serve()

# 현재 루프에서 서버를 태스크로 실행
loop = asyncio.get_event_loop()
loop.create_task(run_server())

print("✅ FastAPI & Gradio 서버가 백그라운드에서 실행 중입니다!")

✅ FastAPI & Gradio 서버가 백그라운드에서 실행 중입니다!


In [12]:
nest_asyncio.apply()